### Connexion à la DB DuckDB

In [1]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Connexion à la DB / Import des Data


In [2]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

IOException: IO Error: Could not set lock on file "/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/amazing.duckdb": Conflicting lock is held in /home/c-enjalbert/miniconda3/bin/python3.12 (PID 12698) by user c-enjalbert. See also https://duckdb.org/docs/connect/concurrency

In [ ]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 4 tables in the database:

1. all_events
2. loaded_files
3. user_events
4. user_segments_kmeans


In [ ]:
all_events_count = con.execute("SELECT COUNT(*) FROM all_events ").fetchone()[0]

print(f"Taille de la table all_events : {all_events_count} logs")

all_events_df = con.execute("SELECT * FROM all_events ORDER BY RANDOM() LIMIT 5000").fetch_df()
print(all_events_df)

Taille de la table all_events : 288779227 logs


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
all_events_df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-17 09:58:14,view,1801691,2232732099754852875,appliances.personal.massager,samsung,487.79,534483166,f0c23cf7-b7e6-4f3e-81f4-b35c3640af94
1,2019-11-05 09:15:23,view,1802024,2053013554415534427,electronics.video.tv,samsung,2573.79,515370667,2a48b2f2-af64-470f-9653-a3ce93da581a
2,2020-01-10 16:11:03,view,100009318,2053013554415534427,electronics.video.tv,None,321.73,599343608,bbd93a82-8fda-4211-a6d7-52226df50c56
3,2019-11-06 18:43:04,view,2701717,2053013563911439225,appliances.kitchen.refrigerators,whirlpool,958.30,536863082,c5779730-d80a-4c20-9792-a502cb2055b7
4,2020-01-13 07:50:46,view,11100214,2232732093857661318,furniture.bedroom.blanket,redmond,28.29,512700668,796faba4-da0c-433e-a347-19c07bd58167
...,...,...,...,...,...,...,...,...,...
4995,2019-10-09 13:03:29,view,1004871,2053013555631882655,electronics.smartphone,samsung,286.51,548173632,19cade4a-f914-35d3-663c-ab2f991b9047
4996,2020-01-20 19:10:19,cart,1004708,2232732093077520756,construction.tools.light,huawei,151.84,564714035,50917657-fbe8-41d4-94ad-a3fc7a38403a
4997,2019-11-14 15:30:39,view,12300918,2053013556311359947,construction.tools.drill,bosch,479.28,514444304,16a9ce75-0788-181e-16e3-b90043bc2221
4998,2020-01-01 09:46:35,view,100032022,2053013565480109009,apparel.shoes.keds,airjordan,136.40,514188289,ddc1789e-33f2-46d3-9709-f8611b3c5c11


In [ ]:
# STEP 0 — Raw Event Data (Input)
# Assuming `all_events_df` contains the raw data
raw_data = all_events_df
# Drop rows where 'category_code' is None or NaN
all_events_df = all_events_df.dropna(subset=["category_code"])

# Verify the result
print(all_events_df)

              event_time event_type product_id          category_id  \
0    2019-12-17 09:58:14       view    1801691  2232732099754852875   
1    2019-11-05 09:15:23       view    1802024  2053013554415534427   
2    2020-01-10 16:11:03       view  100009318  2053013554415534427   
3    2019-11-06 18:43:04       view    2701717  2053013563911439225   
4    2020-01-13 07:50:46       view   11100214  2232732093857661318   
...                  ...        ...        ...                  ...   
4995 2019-10-09 13:03:29       view    1004871  2053013555631882655   
4996 2020-01-20 19:10:19       cart    1004708  2232732093077520756   
4997 2019-11-14 15:30:39       view   12300918  2053013556311359947   
4998 2020-01-01 09:46:35       view  100032022  2053013565480109009   
4999 2020-02-22 07:55:01       view   43700057  2053013559440310913   

                         category_code      brand    price    user_id  \
0         appliances.personal.massager    samsung   487.79  534483166   
1

In [ ]:
# STEP 1 — User → List of Purchased Categories
user_categories = all_events_df.groupby("user_id")["category_code"].apply(list)
print(user_categories)

user_id
293440294              [electronics.smartphone]
399729924          [appliances.kitchen.blender]
431313213                      [apparel.shorts]
453020642    [appliances.kitchen.refrigerators]
476693611                   [apparel.underwear]
                            ...                
621535807               [apparel.shoes.sandals]
621553318        [electronics.audio.microphone]
621559346         [electronics.audio.headphone]
621602671                       [apparel.scarf]
621783399                       [sport.trainer]
Name: category_code, Length: 4086, dtype: object


In [ ]:
# STEP 2 — Hierarchy Expansion
def expand_hierarchy(categories):
    expanded = []
    for category in categories:
        parts = category.split(".")
        expanded.extend([".".join(parts[:i+1]) for i in range(len(parts))])
    return expanded

user_hierarchy = user_categories.apply(expand_hierarchy)
print(user_hierarchy)



user_id
293440294                [electronics, electronics.smartphone]
399729924    [appliances, appliances.kitchen, appliances.ki...
431313213                            [apparel, apparel.shorts]
453020642    [appliances, appliances.kitchen, appliances.ki...
476693611                         [apparel, apparel.underwear]
                                   ...                        
621535807      [apparel, apparel.shoes, apparel.shoes.sandals]
621553318    [electronics, electronics.audio, electronics.a...
621559346    [electronics, electronics.audio, electronics.a...
621602671                             [apparel, apparel.scarf]
621783399                               [sport, sport.trainer]
Name: category_code, Length: 4086, dtype: object


In [ ]:
# STEP 3 — User as “Document” of Tokens
user_tokens = user_hierarchy.apply(lambda x: " ".join(x))
print(user_tokens)

user_id
293440294                   electronics electronics.smartphone
399729924    appliances appliances.kitchen appliances.kitch...
431313213                               apparel apparel.shorts
453020642    appliances appliances.kitchen appliances.kitch...
476693611                            apparel apparel.underwear
                                   ...                        
621535807          apparel apparel.shoes apparel.shoes.sandals
621553318    electronics electronics.audio electronics.audi...
621559346    electronics electronics.audio electronics.audi...
621602671                                apparel apparel.scarf
621783399                                  sport sport.trainer
Name: category_code, Length: 4086, dtype: object


In [ ]:
# STEP 4 — TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(user_tokens)
print(tfidf_matrix.toarray())
print(vectorizer.get_feature_names_out())



[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
['accessories' 'acoustic' 'air_conditioner' 'air_heater' 'alarm' 'apparel'
 'appliances' 'audio' 'auto' 'bag' 'ballet_shoes' 'bath' 'bathroom' 'bed'
 'bedroom' 'belt' 'bicycle' 'blanket' 'blender' 'cabinet' 'camera'
 'carriage' 'cartrige' 'chair' 'climate' 'clocks' 'coffee_grinder'
 'coffee_machine' 'components' 'compressor' 'computers' 'construction'
 'cooler' 'costume' 'country_yard' 'cpu' 'cultivator' 'desktop' 'diapers'
 'dishwasher' 'diving' 'dolls' 'dress' 'drill' 'ebooks' 'electronics'
 'environment' 'espadrilles' 'faucet' 'fmcg' 'fryer' 'furniture'
 'generator' 'grill' 'hair_cutter' 'hammok' 'hdd' 'headphone' 'hob' 'hood'
 'iron' 'ironing_board' 'jeans' 'juicer' 'jumper' 'keds' 'kettle'
 'keyboard' 'kids' 'kitchen' 'lawn_mower' 'light' 'living_room' 'massager'
 'meat_grinder' 'memory' 'microphone' 'microwave' 'mixer' 'moccas

In [ ]:
# STEP 5 — Dimensionality Reduction (SVD)
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=10, random_state=42)
reduced_matrix = svd.fit_transform(tfidf_matrix)
print(reduced_matrix)



[[ 6.22110568e-09  9.66902966e-01  7.10187806e-11 ... -2.32185264e-01
   1.71695773e-07 -1.03446216e-01]
 [ 7.00671022e-04 -8.47261352e-12  8.26429104e-01 ...  1.49720726e-06
   2.06055435e-01  6.48647983e-06]
 [ 2.98788908e-04  3.93880841e-11  3.45770124e-03 ... -1.32294031e-06
  -7.21567240e-04 -6.37750550e-06]
 ...
 [ 4.19622864e-09  6.40351430e-01  8.00279784e-11 ...  6.56619952e-01
   4.30169205e-07 -3.78482819e-01]
 [ 2.97879082e-04  1.02851172e-12  3.44580568e-03 ... -2.12971046e-06
  -7.24979613e-04 -1.09455131e-05]
 [ 3.12346680e-11 -2.11707444e-11 -1.91654149e-09 ... -3.13077541e-06
  -1.65994704e-05 -9.75061003e-06]]


In [ ]:
# STEP 6 — Final Clustering Input
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(reduced_matrix)
print(kmeans.labels_)

# Optional: Alternative Path (Simpler, No TF-IDF)
# Level-1 Only (Coarse)
def level_1_only(categories):
    return [cat.split(".")[0] for cat in categories]

user_level_1 = user_categories.apply(level_1_only)
user_level_1_counts = user_level_1.apply(lambda x: pd.Series(x).value_counts(normalize=True)).fillna(0)
print(user_level_1_counts)

# Interpretation of Clusters
centroids = kmeans.cluster_centers_
print("Cluster centroids:", centroids)

[0 3 2 ... 0 2 2]
           electronics  appliances  apparel  construction  accessories  \
user_id                                                                  
293440294          1.0         0.0      0.0           0.0          0.0   
399729924          0.0         1.0      0.0           0.0          0.0   
431313213          0.0         0.0      1.0           0.0          0.0   
453020642          0.0         1.0      0.0           0.0          0.0   
476693611          0.0         0.0      1.0           0.0          0.0   
...                ...         ...      ...           ...          ...   
621535807          0.0         0.0      1.0           0.0          0.0   
621553318          1.0         0.0      0.0           0.0          0.0   
621559346          1.0         0.0      0.0           0.0          0.0   
621602671          0.0         0.0      1.0           0.0          0.0   
621783399          0.0         0.0      0.0           0.0          0.0   

           computer